In [ ]:

from docplex.mp.model import Model
import pandas as pd
import re

# === Load Nodes from CSV ===
df_nodes = pd.read_csv('nodes_data.csv')  # Columns: id,time,loc
V = df_nodes.to_dict(orient='records')

# === Constants ===
nNodes = len(V)  # Should match CSV rows, e.g., 278
nVehicles = 150
MinUtilTime = 400
MaxUtilTime=500
vehicle_fixed_cost = 10000

freq1 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 2, 0, 0]
freq2 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 4, 0, 0]

H = range(24)
K = range(nVehicles)

# === Arc data ===
# We add arcs between X, HSK, ATB with costs and capacities.
# For feasibility, add arcs inside HSK and inside ATB, and between HSK and ATB.


pattern = re.compile(r"<<(\d+), (\d+), \"(\w+)\">, <(\d+), (\d+), \"(\w+)\">, (\d+), (\d+)>")

E = []
for match in pattern.finditer(data):
    src_id, src_time, src_loc, dst_id, dst_time, dst_loc, cost, capacity = match.groups()
    E.append({
        "src": {"id": int(src_id), "time": int(src_time), "loc": src_loc},
        "dst": {"id": int(dst_id), "time": int(dst_time), "loc": dst_loc},
        "cost": int(cost),
        "cap": int(capacity)
    })

# Identify start and end nodes (assumed here min and max IDs from node list)
id0 = min(node['id'] for node in V)
idend = max(node['id'] for node in V)

# === Model ===
mdl = Model("VehicleScheduling")
mdl.context.cplex_parameters.lpmethod = 3

# Decision variables
x = mdl.binary_var_dict(((e['src']['id'], e['dst']['id'], k) for e in E for k in K), name='x')
z = mdl.binary_var_dict(K, name='z')  # Vehicle usage indicator
T = mdl.continuous_var_dict(K, name='T', lb=0)  # Utilization time per vehicle

vehicle_fixed_cost = 1000
freq_slack_ATB = mdl.continuous_var_dict(H, name='freq_slack_ATB', lb=0)
freq_slack_HSK = mdl.continuous_var_dict(H, name='freq_slack_HSK', lb=0)
freq_penalty = 10000

# Objective
mdl.minimize(
    mdl.sum(e['cost'] * x[e['src']['id'], e['dst']['id'], k] for e in E for k in K)
    + mdl.sum(vehicle_fixed_cost * z[k] for k in K)
    + mdl.sum(freq_penalty * (freq_slack_ATB[h] + freq_slack_HSK[h]) for h in H)
)

# Link x and z
bigM = len(E)
for k in K:
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E) <= bigM * z[k])
    mdl.add_constraint(z[k] <= mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E))

# Flow constraints
for k in K:
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['src']['id'] == id0) == z[k])
    mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['dst']['id'] == idend) == z[k])
    for node in V:
        i = node['id']
        if i != id0 and i != idend:
            mdl.add_constraint(
                mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['src']['id'] == i) ==
                mdl.sum(x[e['src']['id'], e['dst']['id'], k] for e in E if e['dst']['id'] == i)
            )

# Arc capacity constraints
for e in E:
    if e['cap'] == 1:
        mdl.add_constraint(mdl.sum(x[e['src']['id'], e['dst']['id'], k] for k in K) <= 1)

# Utilization time calculation
for k in K:
    mdl.add_constraint(T[k] == mdl.sum(e['cost'] * x[e['src']['id'], e['dst']['id'], k] for e in E if e['cap'] == 1))

# Minimum utilization time per used vehicle
for k in K:
    mdl.add_constraint(T[k] >= MinUtilTime * z[k])
    mdl.add_constraint(T[k] <= MaxUtilTime * z[k])

# Frequency constraints (corrected double loop)
for h in H:
    mdl.add_constraint(
        mdl.sum(
            x[e['src']['id'], e['dst']['id'], k]
            for k in K
            for e in E if e['cap'] == 1 and e['src']['loc'] == 'ATB' and (e['src']['time'] // 60) == h
        )
        + freq_slack_ATB[h] >= freq1[h]
    )
    mdl.add_constraint(
        mdl.sum(
            x[e['src']['id'], e['dst']['id'], k]
            for k in K
            for e in E if e['cap'] == 1 and e['src']['loc'] == 'HSK' and (e['src']['time'] // 60) == h
        )
        + freq_slack_HSK[h] >= freq2[h]
    )

# Solver tuning
mdl.context.cplex_parameters.mip.tolerances.mipgap = 0.01
mdl.context.cplex_parameters.timelimit = 600
mdl.context.cplex_parameters.threads = 16
mdl.context.cplex_parameters.mip.strategy.heuristicfreq = 10

# Solve
solution = mdl.solve(log_output=True)

if solution:
    used_vehicles = [k for k in K if z[k].solution_value > 0.5]
    print(f"Number of vehicles used: {len(used_vehicles)}\n")

    for k in used_vehicles:
        print(f"Schedule for Vehicle {k + 1}: Utilization time = {T[k].solution_value:.2f}")

        current_node = id0

        while current_node != idend:
            next_arcs = [e for e in E if e['src']['id'] == current_node and x[e['src']['id'], e['dst']['id'], k].solution_value > 0.5]
            if not next_arcs:
                print("  ERROR: Route incomplete or disconnected.")
                break
            
            arc = next_arcs[0]
            src = arc['src']
            dst = arc['dst']

            src_h, src_m = divmod(src['time'], 60)
            dst_h, dst_m = divmod(dst['time'], 60)
            print(f"  ({src['loc']} at {src_h:02d}:{src_m:02d}) --> ({dst['loc']} at {dst_h:02d}:{dst_m:02d}), cost={arc['cost']}")

            current_node = dst['id']
        print()
else:
    print("No feasible solution found.")